In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

In [12]:
CLASS_NAMES = [
    "T-shirt/Top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",      "Shirt",   "Sneaker",  "Bag",   "Ankle Boot"
]

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS     = 5
LR_TRAIN   = 0.001
NUM_SEEDS  = 50
STEPS      = 30
LR_DX      = 0.05
LAMBDA     = 1.5
THRESHOLD  = 0.5
SAVE_DIR   = "outputs_fashion"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device     : {DEVICE}")
print(f"PyTorch    : {torch.__version__}")
print(f"Classes    : {CLASS_NAMES}")

Device     : cuda
PyTorch    : 2.10.0+cu128
Classes    : ['T-shirt/Top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot']


In [13]:
class ModelA(nn.Module):
    """
    Model A: 2-block CNN with LeakyReLU.
    LeakyReLU (slope=0.1) keeps small negative activations alive,
    which helps distinguish visually similar clothes (Shirt vs T-shirt).
    """
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64*7*7, 256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.block2(self.block1(x))
        return self.classifier(x.view(x.size(0), -1))





In [14]:

class ModelB(nn.Module):
    """
    Model B: 3-block CNN with BatchNorm + ReLU + Dropout.
    Batch normalization stabilizes training on harder Fashion-MNIST classes.
    """
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128*7*7, 256), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.block3(self.block2(self.block1(x)))
        return self.classifier(x.view(x.size(0), -1))


In [6]:
class ModelC(nn.Module):
    """
    Model C: CNN with a skip (residual-style) connection.
    The skip adds early features back before classification,
    giving richer gradient flow and different decision boundaries.
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2)
        )
        self.skip = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=1),
            nn.AdaptiveAvgPool2d((7, 7))
        )
        self.classifier = nn.Sequential(
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        skip = self.skip(x)
        out  = self.conv2(self.conv1(x))
        out  = out + skip
        return self.classifier(F.relu(out).view(out.size(0), -1))


models_list = [ModelA(), ModelB(), ModelC()]
model_names = ["Model A (LeakyReLU CNN)", "Model B (Deep BN CNN)", "Model C (Skip-connection CNN)"]

for m, n in zip(models_list, model_names):
    params = sum(p.numel() for p in m.parameters())
    print(f"{n:35s} → {params:,} parameters")

Model A (LeakyReLU CNN)             → 824,458 parameters
Model B (Deep BN CNN)               → 1,701,578 parameters
Model C (Skip-connection CNN)       → 421,770 parameters


In [7]:
class NeuronCoverageTracker:
    """
    Tracks neuron activation across ReLU and LeakyReLU layers
    using PyTorch forward hooks — no model code changes needed.
    """
    def __init__(self, model, threshold=0.5):
        self.model           = model
        self.threshold       = threshold
        self.covered_neurons = {}
        self.total_neurons   = {}
        self._hook_handles   = []
        self._register_hooks()

    def _register_hooks(self):
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.ReLU, nn.LeakyReLU)):
                handle = module.register_forward_hook(self._make_hook(name))
                self._hook_handles.append(handle)
                self.covered_neurons[name] = set()

    def _make_hook(self, layer_name):
        def hook(module, input, output):
            with torch.no_grad():
                out = output[0]
                out_flat = out.mean(dim=[1,2]) if out.dim()==3 else out
                if layer_name not in self.total_neurons:
                    self.total_neurons[layer_name] = out_flat.numel()
                active = (out_flat > self.threshold).nonzero(as_tuple=True)[0]
                self.covered_neurons[layer_name].update(active.tolist())
        return hook

    def get_coverage(self):
        total   = sum(self.total_neurons.get(k,0) for k in self.covered_neurons)
        covered = sum(len(v) for v in self.covered_neurons.values())
        return covered/total if total>0 else 0.0

    def get_coverage_percent(self): return self.get_coverage()*100

    def get_uncovered_neurons(self):
        return {
            name: list(set(range(self.total_neurons.get(name,0))) - covered)
            for name, covered in self.covered_neurons.items()
        }

    def reset(self):
        for k in self.covered_neurons: self.covered_neurons[k] = set()

    def remove_hooks(self):
        for h in self._hook_handles: h.remove()
        self._hook_handles = []

    def summary(self):
        print(f"  {'Layer':<35} {'Covered':>8} {'Total':>8} {'%':>8}")
        print("  " + "-"*63)
        for name in self.covered_neurons:
            cov   = len(self.covered_neurons[name])
            total = self.total_neurons.get(name, 0)
            pct   = cov/total*100 if total>0 else 0
            print(f"  {name:<35} {cov:>8} {total:>8} {pct:>7.1f}%")
        print("  " + "-"*63)
        print(f"  {'TOTAL':<35} {'':<8} {'':<8} {self.get_coverage_percent():>7.1f}%")

print("NeuronCoverageTracker defined ✓")

NeuronCoverageTracker defined ✓


In [8]:
class DeepXplore:
    def __init__(self, models, device, lambda_=1.5, threshold=0.5):
        self.models    = models
        self.device    = device
        self.lambda_   = lambda_
        self.threshold = threshold
        self._activations = {}
        for m in self.models:
            m.eval().to(device)

    def _register_hooks(self, model, idx):
        hooks = []
        for name, module in model.named_modules():
            if isinstance(module, (nn.ReLU, nn.LeakyReLU)):
                key = f"m{idx}_{name}"
                def make_hook(k):
                    def hook(mod, inp, out): self._activations[k] = out
                    return hook
                hooks.append(module.register_forward_hook(make_hook(key)))
        return hooks

    def _coverage_loss(self, idx, uncovered_per_layer):
        loss = torch.tensor(0.0, device=self.device)
        for key, activation in self._activations.items():
            if not key.startswith(f"m{idx}_"): continue
            layer = key[len(f"m{idx}_"):]
            uncovered = uncovered_per_layer.get(layer, [])
            if not uncovered: continue
            act = activation.mean(dim=[2,3]) if activation.dim()==4 else activation
            idx_t = torch.tensor(uncovered, device=self.device)
            idx_t = idx_t[idx_t < act.shape[1]]
            if idx_t.numel()==0: continue
            loss = loss + act[0, idx_t].mean()
        return loss

    def _differential_loss(self, outputs):
        probs = torch.stack([F.softmax(o, dim=1) for o in outputs])  # (3,B,10)
        return probs.var(dim=0).sum()

    def _check_disagreement(self, outputs):
        preds = [o.argmax(dim=1).item() for o in outputs]
        return len(set(preds))>1, preds

    def _apply_occlusion(self, x):
        """Block out a random 8×8 patch — simulates partial garment occlusion."""
        with torch.no_grad():
            r = torch.randint(0, 20, (1,)).item()
            c = torch.randint(0, 20, (1,)).item()
            x[:, :, r:r+8, c:c+8] = 0.0
        return x

    def generate(self, seed_input, coverage_trackers,
                 steps=30, lr=0.05, perturbation="noise"):
        x = seed_input.clone().detach().to(self.device).requires_grad_(True)
        optimizer = torch.optim.Adam([x], lr=lr)
        best, found, best_preds = x.clone().detach(), False, []

        for step in range(steps):
            optimizer.zero_grad()
            self._activations.clear()

            all_hooks = []
            for i, m in enumerate(self.models):
                all_hooks.extend(self._register_hooks(m, i))

            outputs = [m(x) for m in self.models]
            for h in all_hooks: h.remove()

            cov_loss  = sum(self._coverage_loss(i, t.get_uncovered_neurons())
                            for i, t in enumerate(coverage_trackers))
            diff_loss = self._differential_loss(outputs)
            loss = -(cov_loss + self.lambda_ * diff_loss)
            loss.backward()

            if perturbation == "brightness":
                with torch.no_grad():
                    x.grad = x.grad.mean() * torch.ones_like(x.grad)
            # "noise" uses raw gradient as-is; "occlusion" handled below

            optimizer.step()
            with torch.no_grad(): x.clamp_(0.0, 1.0)

            if perturbation == "occlusion":
                with torch.no_grad(): x = self._apply_occlusion(x)

            with torch.no_grad():
                disagrees, preds = self._check_disagreement([m(x) for m in self.models])
            if disagrees:
                best, found, best_preds = x.clone().detach(), True, preds
                break

        if not found:
            best = x.clone().detach()
            with torch.no_grad():
                _, best_preds = self._check_disagreement([m(x) for m in self.models])

        return best, found, best_preds

    def run_testing(self, data_loader, coverage_trackers,
                    num_seeds=50, steps=30, lr=0.05):
        bugs, all_gen, seed_count = [], [], 0

        print("\n" + "="*65)
        print("  DeepXplore Testing  —  Fashion-MNIST")
        print("="*65)

        for batch_imgs, batch_labels in data_loader:
            if seed_count >= num_seeds: break
            for i in range(batch_imgs.size(0)):
                if seed_count >= num_seeds: break

                seed  = batch_imgs[i].unsqueeze(0).to(self.device)
                label = batch_labels[i].item()

                for perturb in ["noise", "brightness", "occlusion"]:
                    gen, is_bug, preds = self.generate(
                        seed, coverage_trackers,
                        steps=steps, lr=lr, perturbation=perturb)
                    all_gen.append(gen)
                    if is_bug:
                        bugs.append({
                            "seed_idx"    : seed_count,
                            "true_label"  : label,
                            "true_name"   : CLASS_NAMES[label],
                            "predictions" : preds,
                            "pred_names"  : [CLASS_NAMES[p] for p in preds],
                            "perturbation": perturb,
                            "image"       : gen
                        })

                seed_count += 1
                if seed_count % 10 == 0:
                    covs = [t.get_coverage_percent() for t in coverage_trackers]
                    print(f"  Seeds: {seed_count:3d} | Bugs: {len(bugs):3d} | "
                          f"Cov: A={covs[0]:.1f}% B={covs[1]:.1f}% C={covs[2]:.1f}%")

        final_covs = [t.get_coverage_percent() for t in coverage_trackers]
        print("="*65)
        print(f"  Seeds tested : {seed_count}")
        print(f"  Bugs found   : {len(bugs)}")
        print(f"  Bug rate     : {len(bugs)/max(seed_count,1)*100:.1f}%")
        print(f"  Final Cov → A:{final_covs[0]:.1f}%  B:{final_covs[1]:.1f}%  C:{final_covs[2]:.1f}%")
        print("="*65)

        return {"bugs": bugs, "final_coverages": final_covs,
                "total_seeds": seed_count, "generated_inputs": all_gen}

print("DeepXplore engine defined ✓")

DeepXplore engine defined ✓


In [9]:
FMNIST_MEAN = 0.2860
FMNIST_STD  = 0.3530

class SyntheticFashion(Dataset):
    """Synthetic stand-in for Fashion-MNIST (28x28 grayscale, 10 classes)."""
    def __init__(self, n_samples=6000, img_size=28, n_classes=10, seed=42):
        torch.manual_seed(seed); np.random.seed(seed)
        self.labels = torch.randint(0, n_classes, (n_samples,))
        self.images = self._generate(n_samples, img_size)

    def _generate(self, n, size):
        imgs = torch.zeros(n, 1, size, size)
        for i in range(n):
            c   = self.labels[i].item()
            img = torch.zeros(size, size)
            stripe_top = c * 2 + 2
            r_end = min(stripe_top + 10, size)
            c_end = min(24, size)
            img[stripe_top:r_end, 4:c_end] = 0.6 + 0.3 * torch.rand(r_end - stripe_top, c_end - 4)
            col_pos = 10 + c
            r2 = min(stripe_top + 6, size)
            c2 = min(col_pos + 4, size)
            if r2 > stripe_top and c2 > col_pos:
                img[stripe_top:r2, col_pos:c2] = 0.9
            img += 0.08 * torch.randn(size, size)
            img = (img.clamp(0, 1) - FMNIST_MEAN) / FMNIST_STD
            imgs[i, 0] = img
        return imgs

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]


def get_dataloaders():
    try:
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((FMNIST_MEAN,), (FMNIST_STD,))
        ])
        train_ds = datasets.FashionMNIST("./data", train=True,  download=True, transform=tf)
        test_ds  = datasets.FashionMNIST("./data", train=False, download=True, transform=tf)
        print("Fashion-MNIST loaded ✓")
    except Exception as e:
        print(f"Download unavailable ({e}). Using synthetic data.")
        train_ds = SyntheticFashion(n_samples=6000)
        test_ds  = SyntheticFashion(n_samples=1000)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"Train: {len(train_ds):,} samples  |  Test: {len(test_ds):,} samples")
    return train_loader, test_loader

train_loader, test_loader = get_dataloaders()

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 166kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.07MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.7MB/s]

Fashion-MNIST loaded ✓
Train: 60,000 samples  |  Test: 10,000 samples


In [10]:
def train_model(model, loader, name):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR_TRAIN)
    criterion = nn.CrossEntropyLoss()
    history   = []
    print(f"\nTraining {name}...")
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        avg = total_loss/len(loader)
        history.append(avg)
        print(f"  Epoch {epoch}/{EPOCHS}  Loss: {avg:.4f}")
    return history

def evaluate(model, loader, name):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (model(imgs).argmax(1)==labels).sum().item()
            total   += labels.size(0)
    acc = correct/total*100
    print(f"  {name}: {acc:.2f}%")
    return acc

# Rebuild models fresh
model_A = ModelA(); model_B = ModelB(); model_C = ModelC()
models_list = [model_A, model_B, model_C]

histories  = [train_model(m, train_loader, n) for m,n in zip(models_list, model_names)]
print("\nTest Accuracy:")
accuracies = [evaluate(m, test_loader, n)     for m,n in zip(models_list, model_names)]


Training Model A (LeakyReLU CNN)...
  Epoch 1/5  Loss: 0.4059
  Epoch 2/5  Loss: 0.2666
  Epoch 3/5  Loss: 0.2199
  Epoch 4/5  Loss: 0.1910
  Epoch 5/5  Loss: 0.1677

Training Model B (Deep BN CNN)...
  Epoch 1/5  Loss: 0.4540
  Epoch 2/5  Loss: 0.3020
  Epoch 3/5  Loss: 0.2592
  Epoch 4/5  Loss: 0.2280
  Epoch 5/5  Loss: 0.2050

Training Model C (Skip-connection CNN)...
  Epoch 1/5  Loss: 0.4459
  Epoch 2/5  Loss: 0.2883
  Epoch 3/5  Loss: 0.2459
  Epoch 4/5  Loss: 0.2129
  Epoch 5/5  Loss: 0.1891

Test Accuracy:
  Model A (LeakyReLU CNN): 92.07%
  Model B (Deep BN CNN): 92.36%
  Model C (Skip-connection CNN): 91.03%


In [15]:
# Training loss curves
fig, ax = plt.subplots(figsize=(8,4))
colors = ["#4C72B0", "#DD8452", "#55A868"]
for hist, name, col in zip(histories, ["Model A","Model B","Model C"], colors):
    ax.plot(range(1, len(hist)+1), hist, marker='o', color=col, label=name)
ax.set_title("Fashion-MNIST — Training Loss per Epoch", fontsize=13)
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-Entropy Loss")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120)
plt.show()
print("Training curves saved ✓")

Training curves saved ✓


In [16]:
# Setup trackers
trackers = [NeuronCoverageTracker(m, threshold=THRESHOLD) for m in models_list]

# Warm-up: one test batch
print("Warming up neuron coverage...")
for model, tracker in zip(models_list, trackers):
    model.eval()
    with torch.no_grad():
        for imgs, _ in test_loader:
            _ = model(imgs.to(DEVICE)); break

for tracker, name in zip(trackers, ["Model A","Model B","Model C"]):
    print(f"  {name} warm-up coverage: {tracker.get_coverage_percent():.1f}%")

Warming up neuron coverage...
  Model A warm-up coverage: 19.0%
  Model B warm-up coverage: 4.0%
  Model C warm-up coverage: 15.2%


In [17]:
# Run DeepXplore
dx = DeepXplore(models_list, device=DEVICE, lambda_=LAMBDA, threshold=THRESHOLD)

results = dx.run_testing(
    test_loader, trackers,
    num_seeds=NUM_SEEDS,
    steps=STEPS,
    lr=LR_DX
)


  DeepXplore Testing  —  Fashion-MNIST
  Seeds:  10 | Bugs:  29 | Cov: A=71.9% B=27.1% C=38.4%
  Seeds:  20 | Bugs:  57 | Cov: A=73.3% B=28.7% C=38.8%
  Seeds:  30 | Bugs:  87 | Cov: A=73.3% B=29.0% C=38.8%
  Seeds:  40 | Bugs: 114 | Cov: A=73.3% B=29.2% C=38.8%
  Seeds:  50 | Bugs: 143 | Cov: A=73.3% B=29.4% C=38.8%
  Seeds tested : 50
  Bugs found   : 143
  Bug rate     : 286.0%
  Final Cov → A:73.3%  B:29.4%  C:38.8%


In [18]:
# Per-layer coverage breakdown
for tracker, name in zip(trackers, ["Model A","Model B","Model C"]):
    print(f"\n{name}")
    tracker.summary()


Model A
  Layer                                Covered    Total        %
  ---------------------------------------------------------------
  block1.1                                   1       32     3.1%
  block2.1                                   1       64     1.6%
  classifier.1                             256      256   100.0%
  ---------------------------------------------------------------
  TOTAL                                                    73.3%

Model B
  Layer                                Covered    Total        %
  ---------------------------------------------------------------
  block1.2                                  16       32    50.0%
  block2.2                                  34       64    53.1%
  block3.2                                  11      128     8.6%
  classifier.1                              80      256    31.2%
  ---------------------------------------------------------------
  TOTAL                                                    29.4%

Mo

In [19]:
# Neuron Coverage Bar Chart
coverages = [t.get_coverage_percent() for t in trackers]
colors    = ["#4C72B0","#DD8452","#55A868"]
labels_ax = ["Model A","Model B","Model C"]

fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(labels_ax, coverages, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, coverages):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_title("Neuron Coverage After DeepXplore — Fashion-MNIST", fontsize=13)
ax.set_ylabel("Coverage (%)")
ax.axhline(100, color='red', linestyle='--', alpha=0.4, label='Max (100%)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/neuron_coverage.png", dpi=120)
plt.show()

In [20]:
# Bug-Triggering Input Examples (with clothing class names)
bugs = results["bugs"]

if not bugs:
    print("No disagreements found — try increasing NUM_SEEDS or STEPS.")
else:
    n   = min(len(bugs), 6)
    fig = plt.figure(figsize=(14, 3.2*n))
    gs  = gridspec.GridSpec(n, 3, figure=fig, wspace=0.05, hspace=0.65)

    for row, bug in enumerate(bugs[:n]):
        img = bug["image"].squeeze().cpu().numpy() * FMNIST_STD + FMNIST_MEAN
        img = np.clip(img, 0, 1)
        preds, pred_names = bug["predictions"], bug["pred_names"]
        mnames = ["Model A","Model B","Model C"]

        for col in range(3):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
            is_disagree = len(set(preds)) > 1
            color = "red" if is_disagree else "green"
            ax.set_title(
                f"{mnames[col]}\n→ {pred_names[col]}",
                fontsize=8.5, color=color
            )
            ax.axis('off')

        # True label annotation
        fig.text(0.01, 1-(row+0.5)/n,
                 f"True:\n{bug['true_name']}\n({bug['perturbation']})",
                 fontsize=7, transform=fig.transFigure,
                 va='center', color='navy')

    fig.suptitle(
        "Bug-Triggering Inputs — Fashion-MNIST\n"
        "Red = models disagree on clothing category",
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/bug_examples.png", dpi=120, bbox_inches='tight')
    plt.show()
    print(f"\nTotal bugs found: {len(bugs)}")

/tmp/ipykernel_3818/2510304931.py:39: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Total bugs found: 143


In [21]:
# Summary Dashboard
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].bar(["Model A","Model B","Model C"],
            results["final_coverages"], color=colors, edgecolor='white')
axes[0].set_title("Final Neuron Coverage (%)"); axes[0].set_ylim(0, 110)
for i, v in enumerate(results["final_coverages"]):
    axes[0].text(i, v+1, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

total  = results["total_seeds"]; bugs_n = len(results["bugs"])
axes[1].axis('off')
table_data = [
    ["Dataset",       "Fashion-MNIST"],
    ["Seeds tested",  str(total)],
    ["Bugs found",    str(bugs_n)],
    ["Bug rate",      f"{bugs_n/max(total,1)*100:.1f}%"],
    ["Model A Acc",   f"{accuracies[0]:.1f}%"],
    ["Model B Acc",   f"{accuracies[1]:.1f}%"],
    ["Model C Acc",   f"{accuracies[2]:.1f}%"],
    ["Perturbations", "noise / brightness / occlusion"],
]
tbl = axes[1].table(cellText=table_data, colLabels=["Metric","Value"],
                    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
axes[1].set_title("DeepXplore Results — Fashion-MNIST")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/summary_dashboard.png", dpi=120)
plt.show()

In [23]:
# Bug Detail Table
bugs = results["bugs"]
if bugs:
    print(f"{'#':>3}  {'True Class':<16} {'Model A':<14} {'Model B':<14} {'Model C':<14} Perturbation")
    print("-"*80)
    for i, bug in enumerate(bugs[:10]):
        p = bug['predictions']
        flag = "⚠ DISAGREE" if len(set(p))>1 else ""
        print(f"{i+1:>3}  {bug['true_name']:<16} "
              f"{CLASS_NAMES[p[0]]:<14} {CLASS_NAMES[p[1]]:<14} {CLASS_NAMES[p[2]]:<14} "
              f"{bug['perturbation']}  {flag}")

for t in trackers: t.remove_hooks()
print(f"\nAll outputs saved to ./{SAVE_DIR}/")
print("Done ✓")

  #  True Class       Model A        Model B        Model C        Perturbation
--------------------------------------------------------------------------------
  1  Ankle Boot       Ankle Boot     Ankle Boot     Bag            noise  ⚠ DISAGREE
  2  Ankle Boot       Ankle Boot     Ankle Boot     Bag            brightness  ⚠ DISAGREE
  3  Ankle Boot       Ankle Boot     Bag            Bag            occlusion  ⚠ DISAGREE
  4  Pullover         Pullover       Pullover       Shirt          noise  ⚠ DISAGREE
  5  Pullover         Pullover       Pullover       Bag            brightness  ⚠ DISAGREE
  6  Pullover         Coat           Pullover       Pullover       occlusion  ⚠ DISAGREE
  7  Trouser          Trouser        T-shirt/Top    Trouser        noise  ⚠ DISAGREE
  8  Trouser          Trouser        T-shirt/Top    Trouser        brightness  ⚠ DISAGREE
  9  Trouser          Trouser        T-shirt/Top    Trouser        occlusion  ⚠ DISAGREE
 10  Trouser          Trouser        T-shirt/To